In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from tqdm import tqdm
from sklearn.feature_selection import SelectKBest, f_regression

In [ ]:
# Load cleaned + engineered data
df_train = pd.read_parquet("/content/drive/MyDrive/processed_data/train.parquet")
df_val = pd.read_parquet("/content/drive/MyDrive/processed_data/val.parquet")
df_test = pd.read_parquet("/content/drive/MyDrive/processed_data/test.parquet")

print(f"Number of rows in df_train: {len(df_train)}")
df_train.head()

Number of rows in df_train: 4215569


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,price_diff_vs_visitor_hist,prop_click_rate,prop_book_rate,prop_mean_position,prop_median_price,prop_n_impressions,dest_click_rate,dest_book_rate,dest_median_price,relevance
0,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,893,3,3.5,...,0.0,0.026854,0.676630,25.149622,118.000000,528,0.034722,0.758361,119.0,0
1,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,10404,4,4.0,...,0.0,0.030528,0.609565,22.723791,129.000000,496,0.034722,0.758361,119.0,0
2,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,21315,3,4.5,...,0.0,0.007273,0.556856,23.935345,168.920013,464,0.034722,0.758361,119.0,0
3,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,27348,2,4.0,...,0.0,0.024100,0.591533,24.353403,65.059998,382,0.034722,0.758361,119.0,0
4,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,29604,4,3.5,...,0.0,0.049560,0.637872,12.907802,118.000000,564,0.034722,0.758361,119.0,0


In [ ]:
# Define features, X, and y
target_col = "relevance"
exclude_cols = ["click_bool", "booking_bool", "gross_bookings_usd", "position", "relevance"]
id_cols = ["srch_id", "date_time", "prop_id"]
feature_cols = [c for c in df_train.columns if c not in exclude_cols + id_cols]

X_train, y_train = df_train[feature_cols], df_train["relevance"]
X_val, y_val = df_val[feature_cols], df_val["relevance"]
X_test = df_test[feature_cols]

In [ ]:
# Feature selection
selector = SelectKBest(score_func=f_regression, k=20)
selector.fit(X_train.fillna(0), y_train)

selected_features = np.array(feature_cols)[selector.get_support()]
print(f"Selected features:\n{selected_features}")

X_train = X_train[selected_features]
X_val   = X_val[selected_features]
X_test  = X_test[selected_features]

In [ ]:
# Define helper function for NDCG@5 score

def get_cdg(scores):
  return sum(r / np.log2(i + 2) for i, r in enumerate(scores))

def get_ndcg(df):
  scores = []

  for _, group in df.groupby("srch_id"):
    group = group.sort_values("rank")
    actual = group["relevance"].values[::-1][:k]
    ideal = np.sort(group["relevance"].values)[::-1][:k]

    idcg = get_cdg(ideal)
    if idcg == 0:
      continue
    else: scores.append(get_cdg(actual) / idcg)

    return np.mean(scores)


In [ ]:
# Tune k = # groups
scaler = StandardScaler()
X_train_scaled, X_val_scaled = scaler.fit_transform(X_train), scaler.fit_transform(X_val)

best_k = k, best_ndcg = None, -np.inf

for k in tqdm([5, 10, 25, 50, 100]):
    knn = KNeighborsRegressor( # Justify parameter choices
        n_neighbors = k,
        weights = 'distance',
        metric = 'euclidean',
        algorithm = 'ball_tree',
        n_jobs = -1
    )

    knn.fit(X_train_scaled, y_train)

    df_val_copy = df_val.copy()
    df_val_copy["pred_relevance"] = knn.predict(X_val_scaled)
    df_val_copy["rank"] = df_val_copy.groupby("srch_id")["pred_relevance"].rank(ascending = False, method = "first")

    ndcg = get_ndcg(df_val_copy)

    print(f"Best k = {best_k},   Best NDCG@5 = {best_ndcg:.3f}")

    if ndcg > best_ndcg:
        best_ndcg = ndcg
        best_k = k

print(f"Best k = {best_k},   Best NDCG@5 = {best_ndcg:.3f}")


 17%|█▋        | 1/6 [00:07<00:35,  7.11s/it]

Best k = (None, -inf),   Best NDCG@5 = -inf


 33%|███▎      | 2/6 [00:15<00:31,  7.76s/it]

Best k = 1,   Best NDCG@5 = 0.000


 50%|█████     | 3/6 [00:21<00:20,  6.88s/it]

...

Best k = 5,   Best NDCG@5 = 0.631


 83%|████████▎ | 5/6 [00:35<00:07,  7.17s/it]

Best k = 5,   Best NDCG@5 = 0.631


100%|██████████| 6/6 [00:42<00:00,  7.14s/it]

Best k = 5,   Best NDCG@5 = 0.631
Best k = 5,   Best NDCG@5 = 0.631


In [ ]:
# Fit final model w/ best k
df = pd.concat([df_train, df_val], ignore_index = True)
X = df[feature_cols]
y = df["relevance"]

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor( # Keeping same params as before
        n_neighbors = best_k,
        weights = 'distance',
        metric = 'euclidean',
        algorithm = 'ball_tree',
        n_jobs = -1
    ))
])

knn_pipeline.fit(X, y)

Pipeline(steps=[('scaler', StandardScaler()),
                ('knn',
                 KNeighborsRegressor(algorithm='ball_tree', metric='euclidean',
                                     n_jobs=-1, weights='distance'))])

In [ ]:
# Generate submission
df_test_copy = df_test.copy()
df_test_copy["pred_relevance"] = knn_pipeline.predict(X_test)

submission = (
    df_test_copy
    .sort_values(["srch_id", "pred_relevance"], ascending = [True, False])[["srch_id", "prop_id"]]
    .rename(columns={"srch_id": "SearchId", "prop_id": "PropertyId"})
)

submission.to_csv("submission_knn.csv", index = False)

KeyboardInterrupt: 